# Last Layer Fine-Tuning SigLIP2-SO400M — folds_v3
Training last-layer FT, simpan model terbaik ke Drive, lalu evaluasi di data test mentah.

In [ ]:
import os, sys, shutil
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# 2. Clone atau Pull Repo Terbaru
REPO_DIR = "/content/satria-data-bdcugm02"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

# 3. Install dependensi
os.system("pip install -q -U 'torchao>=0.16.0'")
os.system(f"pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt")

# 4. Copy gambar train ke /tmp untuk I/O cepat
DRIVE_TRAIN_DIR = "/content/drive/MyDrive/BDC2026/train"
LOCAL_TRAIN_DIR = "/tmp/dataset/train"

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TRAIN_DIR):
    print("Memulai copy gambar ke /tmp dengan 32 workers...")
    all_imgs = [
        os.path.join(r, f)
        for r, _, fs in os.walk(DRIVE_TRAIN_DIR)
        for f in fs if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ]
    pairs = [(src, os.path.join(LOCAL_TRAIN_DIR, os.path.relpath(src, DRIVE_TRAIN_DIR))) for src in all_imgs]
    with ThreadPoolExecutor(max_workers=32) as ex:
        list(ex.map(copy_img_worker, pairs))
    print(f"Selesai: {len(pairs)} gambar dicopy ke {LOCAL_TRAIN_DIR}")
else:
    print("Folder gambar Drive tidak ditemukan.")


## Training — Last Layer (4 blok terakhir dibuka)
Model terbaik **otomatis disimpan ke Drive** setiap val_f1 meningkat.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/satria-data-bdcugm02/track_a/src")
sys.path.insert(0, "/content/satria-data-bdcugm02/track_b/src")
sys.path.insert(0, "/content/satria-data-bdcugm02/track_b/experiments")

import embed, lora_ft, config
importlib.reload(config); importlib.reload(embed); importlib.reload(lora_ft)
from config import CFG, make_cfg

VARIANT    = "last_layer"
CHECKPOINT = "google/siglip2-so400m-patch14-384"

cfg = make_cfg(
    run_name    = f"{VARIANT}_ft_fold0_5ep_v3",
    folds_csv   = CFG.folds_v3_csv,
    batch       = 8,
    accum_steps = 4,
)

# Training -- model terbaik otomatis disimpan ke Drive tiap val_f1 naik
result = lora_ft.run_smoke_test_fold0(
    variant       = VARIANT,
    cfg           = cfg,
    checkpoint    = CHECKPOINT,
    max_epochs    = 5,
    n_last_blocks = 4,
)
result


## Konfirmasi Checkpoint
Cek file model yang disimpan di Drive.

In [ ]:
import os
ckpt_path = result["best_ckpt_path"]
best_epoch = result["best_epoch"]
best_f1 = result["best_val_f1"]

if ckpt_path and os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print("Checkpoint tersimpan!")
    print(f"  Path       : {ckpt_path}")
    print(f"  Ukuran     : {size_mb:.1f} MB")
    print(f"  Best Epoch : {best_epoch}")
    print(f"  Best val_f1: {best_f1:.4f}")
else:
    print("Checkpoint tidak ditemukan. Cek log training di atas.")


## Evaluasi di Data Test Mentah
Load checkpoint terbaik, inference di semua gambar test, simpan prediksi ke Drive.

In [ ]:
import importlib
importlib.reload(lora_ft)

# Ambil otomatis dari training. Atau isi manual:
# CKPT_PATH = "/content/drive/MyDrive/BDC2026apace/output_trackB/last_layer_ft_fold0_5ep_v3_best.pt"
CKPT_PATH = result["best_ckpt_path"]

filepaths, preds, probs = lora_ft.evaluate_on_test(
    ckpt_path  = CKPT_PATH,
    cfg        = cfg,
    batch_size = 32,
)


In [ ]:
import pandas as pd
import os

CLASS_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

df_pred = pd.DataFrame({
    "filepath"        : filepaths,
    "filename"        : [os.path.basename(p) for p in filepaths],
    "pred_label_id"   : preds,
    "pred_label_name" : [CLASS_NAMES[p] for p in preds],
    "prob_Recyclable" : [p[0] for p in probs],
    "prob_Electronic" : [p[1] for p in probs],
    "prob_Organic"    : [p[2] for p in probs],
    "confidence"      : [max(p) for p in probs],
})

DRIVE_BASE = "/content/drive/MyDrive/BDC2026apace"
out_path   = f"{DRIVE_BASE}/output_trackB/predictions_lastlayer_v3.csv"
df_pred.to_csv(out_path, index=False)
total = len(df_pred)
print(f"Prediksi disimpan: {out_path}")
print(f"Total: {total} gambar")
df_pred[["filename", "pred_label_name", "confidence"]].head(10)
